# Chronic Kidney Disease (CKD) KNN Starter Notebook

This notebook demonstrates a basic end-to-end workflow for a KNN classifier that predicts CKD from a CSV dataset.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.preprocessing import clean_ckd_dataframe, load_dataset, prepare_features_and_target

In [ ]:
dataset_path = Path('../data/chronic_kidney_disease.csv')
if not dataset_path.exists():
    raise FileNotFoundError(
        f'Missing dataset: {dataset_path}. Place chronic_kidney_disease.csv in the data folder.'
    )

raw_df = load_dataset(dataset_path)
raw_df.head()

In [ ]:
clean_df = clean_ckd_dataframe(raw_df)
clean_df.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
X, y = prepare_features_and_target(clean_df, target_col='classification')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier()),
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan'],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
best_model

In [ ]:
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {accuracy:.3f}')

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap='Blues')
plt.title('KNN Confusion Matrix')
plt.show()

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
results['n_neighbors'] = results['param_knn__n_neighbors'].astype(int)
results['weights'] = results['param_knn__weights'].astype(str)
results['metric'] = results['param_knn__metric'].astype(str)

plot_df = results[results['metric'] == 'euclidean']

plt.figure(figsize=(8, 5))
for weight, grp in plot_df.groupby('weights'):
    grp = grp.sort_values('n_neighbors')
    plt.plot(grp['n_neighbors'], grp['mean_test_score'], marker='o', label=f'weights={weight}')

plt.xlabel('Number of Neighbors (k)')
plt.ylabel('Mean CV Accuracy')
plt.title('KNN Grid Search Results (metric=euclidean)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()